# Exploración Inicial del Dataset — MVA Hackathon 2026

**Track 2 — Reposicionamiento de Fármacos**  
Dataset: `SageBio/mva-hackathon-2026-data`

Este notebook documenta la exploración inicial del dataset para que todos los integrantes del equipo partan del mismo punto. Cubre:

1. Verificación del entorno y autenticación
2. Archivos disponibles y sus tamaños
3. Fenotipo clínico del paciente (HPO)
4. Estructura y estadísticas del VCF
5. Estrategia de análisis propuesta

---
> **Prerequisitos:** haber completado la autenticación en HuggingFace (`python scripts/hf_login.py`) y descargado los archivos (`python scripts/download_data.py --file ...`). Ver el README.

## 0. Verificar entorno

In [ ]:
import sys
from pathlib import Path
from huggingface_hub import whoami

print(f"Python: {sys.version}")

# Verificar autenticación HF
try:
    info = whoami()
    print(f"HuggingFace: autenticada como {info['name']} ✓")
except Exception:
    print("⚠️  No autenticada en HuggingFace. Ejecuta: python scripts/hf_login.py")

# Verificar archivos descargados
DATA_DIR = Path("../data/raw")
expected = [
    "Challenge_Clinical_Phenotype_1.docx",
    "WGS_EX2312012_HGWCNDSX7.vcf.gz",
    "WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi",
]
for fname in expected:
    p = DATA_DIR / fname
    status = f"{p.stat().st_size / 1e6:.1f} MB ✓" if p.exists() else "❌ NO ENCONTRADO"
    print(f"  {fname}: {status}")

## 1. Archivos disponibles en el dataset

El dataset `SageBio/mva-hackathon-2026-data` contiene 13 archivos (~85 GB total).

**Para Track 2 (reposicionamiento) descargamos solo:**

| Archivo | Tamaño | Uso |
|---------|--------|-----|
| `Challenge_Clinical_Phenotype_1.docx` | ~0 MB | Fenotipo HPO del paciente |
| `WGS_EX2312012_HGWCNDSX7.vcf.gz` | 300.6 MB | Variantes genómicas (WGS) |
| `WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi` | 2.2 MB | Índice tabix del VCF |

Los 8 archivos FASTQ (~79 GB) son lecturas crudas — solo necesarios si se quiere rehacer el alineamiento desde cero, lo cual está fuera del alcance del Track 2.

## 2. Fenotipo Clínico del Paciente

In [ ]:
from docx import Document
import pandas as pd

doc = Document(DATA_DIR / "Challenge_Clinical_Phenotype_1.docx")

# Extraer la tabla HPO
table = doc.tables[0]
rows = [[cell.text.strip() for cell in row.cells] for row in table.rows]
df_hpo = pd.DataFrame(rows[1:], columns=rows[0])
df_hpo

### Interpretación del fenotipo

La constelación de rasgos — **rabdomiosarcoma + nefrocalcinosis + RCIU severo + prematuridad + abortos recurrentes parentales** — es altamente sugestiva de **Aneuploidy Variegada en Mosaico (MVA)**.

**Genes candidatos principales** (autosómico recesivo):
- **BUB1B** (~50% de casos MVA): codifica BUBR1, componente clave del Spindle Assembly Checkpoint (SAC)
- **CEP57**: centrosoma, asociado a MVA tipo 2
- **TRIP13**: ATPasa que regula el complejo MCC (Mitotic Checkpoint Complex)

**Mecanismo general:** variante LOF en gen SAC → inestabilidad cromosómica severa → mosaicismo aneuplo → fenotipo MVA

**Patrón de herencia esperado en el VCF:** variante homocigótica o heterocigótica compuesta en uno de estos genes.

## 3. Estructura del VCF

In [ ]:
import gzip

VCF = DATA_DIR / "WGS_EX2312012_HGWCNDSX7.vcf.gz"

meta_lines = []
with gzip.open(VCF, "rt") as f:
    for line in f:
        if not line.startswith("#"):
            break
        meta_lines.append(line.strip())

# Mostrar líneas clave del encabezado
important_tags = ("fileformat", "GATKCommand", "SentieonCommand", "reference", "#CHROM")
for l in meta_lines:
    if any(tag in l for tag in important_tags):
        print(l[:200])

**Información clave del encabezado:**
- Formato: VCFv4.2
- Muestra única: `WGS_EX2312012` (un solo paciente)
- Pipeline de variant calling: **Sentieon Haplotyper** (compatible con GATK HaplotypeCaller)
- Filtrado: GATK VariantFiltration con filtros estándar (QD2, MQ40, FS60, MQRankSum-12.5, RPRS-8)
- Referencia: **GRCh38** (sin prefijo `chr` — cromosomas nombrados `1`, `2`, etc.)
- Anotación dbSNP138
- **Fecha de llamado de variantes:** 5 de febrero de 2025

## 4. Estadísticas del VCF

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

filters = Counter()
variant_types = Counter()
chroms = Counter()

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t", 8)
        ref, alt, filt = parts[3], parts[4], parts[6]
        chrom = parts[0]
        filters[filt] += 1
        if len(ref) == 1 and len(alt) == 1:
            variant_types["SNV"] += 1
        else:
            variant_types["INDEL"] += 1
        chroms[chrom] += 1

total = sum(filters.values())
print(f"Total variantes: {total:,}")
print(f"  PASS: {filters['PASS']:,} ({filters['PASS']/total*100:.1f}%)")
print(f"  Filtradas: {total - filters['PASS']:,} ({(total-filters['PASS'])/total*100:.1f}%)")
print(f"\nPor tipo:")
for k, v in variant_types.items():
    print(f"  {k}: {v:,} ({v/total*100:.1f}%)")

In [ ]:
# Visualización: distribución por filtro y tipo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Filtros
top_filters = dict(filters.most_common(7))
axes[0].barh(list(top_filters.keys()), list(top_filters.values()), color="steelblue")
axes[0].set_title("Variantes por filtro GATK")
axes[0].set_xlabel("Número de variantes")

# Tipo
axes[1].pie(
    list(variant_types.values()),
    labels=list(variant_types.keys()),
    autopct="%1.1f%%",
    colors=["#4C72B0", "#DD8452"],
)
axes[1].set_title("SNV vs INDEL")

plt.tight_layout()
plt.savefig("../results/vcf_stats_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en results/vcf_stats_overview.png")

**Resumen VCF:**
- **5,012,204 variantes totales**
- **4,740,790 pasan filtros de calidad (94.6%)** → estas son las que analizamos
- 81.4% SNVs · 18.6% INDELs
- Cobertura genómica completa (todos los cromosomas representados)

## 5. Estrategia de Análisis — Track 2

```
VCF (5M variantes)
        │
        ▼
Filtrar por calidad (PASS) + genes candidatos MVA
(BUB1B, CEP57, TRIP13, MAD1L1, MAD2L2)
        │
        ▼
Priorizar por frecuencia poblacional (gnomAD AF < 0.01)
+ impacto funcional (LOF: frameshift/nonsense/splice)
+ missense con CADD ≥ 20
        │
        ▼
Identificar variante causante
(homocigótica o heterocigótica compuesta)
        │
        ▼
Caracterizar mecanismo biológico
(gen afectado → vía SAC/centrosoma → inestabilidad cromosómica)
        │
        ▼
Buscar fármacos reposicionables
(OpenTargets / ChEMBL / DrugBank)
que actúen sobre la vía afectada
        │
        ▼
Informe final + video pitch
```

### Siguiente notebook: `02_filtrado_vcf_genes_mva.ipynb`

Filtrará el VCF para quedarse solo con variantes en genes candidatos y aplicará criterios de priorización.